In [ ]:
# Build the class dataset: the full Parquet (benign + attack) converted to one CSV.
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
         .master("local[*]")
         .appName("build_class_dataset")
         .config("spark.driver.memory", "8g")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.session.timeZone", "UTC")

base = ("hdfs://192.168.1.250:9000/tmp/82913-CNT4416-202608/correlated_dataset/"
        "2026-09-26_00-28-45/finalized_data")
out  = "hdfs://192.168.1.250:9000/tmp/82913-CNT4416-202608/class_dataset_full_csv"

# Column order taken from the sample CSV header, so the file matches what the notebook expects
cols = spark.read.option("header", True).csv(base + "/Sample_Dataset_CSV/Benign/part-*.csv").columns

full = spark.read.parquet(base + "/Full_Dataset_Parquet/2026-*")
full = full.withColumn("datetime", F.date_format("datetime", "yyyy-MM-dd'T'HH:mm:ss.SSS'Z'"))
full = full.select([(F.col(c).cast("string") if c in full.columns
                     else F.lit(None).cast("string")).alias(c) for c in cols])

print("Total rows:", full.count())
full.groupBy("label_tactic").count().orderBy("count", ascending=False).show(truncate=False)

full.coalesce(1).write.mode("overwrite").option("header", True).csv(out)
print("Written to", out)

In [1]:
# pull attack output to compare to mission logs
from pyspark.sql import SparkSession, functions as F
spark = SparkSession.builder.master("local[*]").config("spark.driver.memory", "8g").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.session.timeZone", "UTC")

df = (spark.read.option("header", True)
      .csv("hdfs://192.168.1.250:9000/tmp/82913-CNT4416-202608/class_dataset_full_csv"))
summary = (df.filter((F.col("label_tactic") != "none") & (F.col("label_technique") != "Duplicate"))
             .withColumn("day", F.substring("datetime", 1, 10))
             .groupBy("src_ip_zeek", "dest_ip_zeek", "label_technique", "label_tactic", "day")
             .agg(F.count(F.lit(1)).alias("connections"),
                  F.countDistinct("dest_port_zeek").alias("distinct_dest_ports"),
                  F.min("datetime").alias("first"), F.max("datetime").alias("last"))
             .orderBy("first"))
pdf = summary.toPandas()
pdf.to_csv("/home/ubuntu/CNT4416_attack_summary.csv", index=False)
print(len(pdf), "rows")
#

26/10/26 16:57:35 WARN Utils: Your hostname, car-ubuntu resolves to a loopback address: 127.0.1.1; using 192.168.1.131 instead (on interface ens33)
26/10/26 16:57:35 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/26 16:57:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/26 16:57:37 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/10/26 16:57:37 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
                                                                                

284 rows


In [ ]:
#test missing entries
from datetime import datetime
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.master("local[*]").config("spark.driver.memory", "6g").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.session.timeZone", "UTC")

df = (spark.read.option("header", True)
      .csv("hdfs://192.168.1.250:9000/tmp/82913-CNT4416-202608/class_dataset_full_csv")
      .select("src_ip_zeek", "dest_ip_zeek", "dest_port_zeek", "label_technique",
              F.col("ts").cast("double").alias("ts"))
      .cache())

# (log line, source IP, destination IP or subnet prefix, start, end, logged port)
checks = [
    (20, "143.88.3.10",  "143.88.3.12",  "2026-09-05T12:36:08Z", "2026-09-05T12:36:15Z", "3389"),
    (21, "143.88.1.10",  "143.88.1.11",  "2026-09-08T14:40:00Z", "2026-09-08T15:05:00Z", "8888"),
    (22, "143.88.4.13",  "143.88.4.11",  "2026-09-06T14:12:00Z", "2026-09-06T14:12:01Z", "80"),
    (29, "143.88.3.10",  "143.88.4.11",  "2026-09-09T13:59:00Z", "2026-09-09T13:59:10Z", "445"),
    (30, "143.88.3.10",  "143.88.4.12",  "2026-09-09T14:41:00Z", "2026-09-09T14:42:20Z", "3389"),
    (31, "143.88.6.10",  "143.88.7.",    "2026-09-09T16:14:00Z", "2026-09-09T16:16:00Z", "1-32768"),
    (50, "143.88.4.10",  "143.88.5.11",  "2026-09-17T01:35:53Z", "2026-09-17T01:35:53Z", "21"),
    (51, "143.88.7.10",  "143.88.8.11",  "2026-09-17T22:03:47Z", "2026-09-17T22:04:01Z", "21"),
    (52, "143.88.7.10",  "143.88.8.12",  "2026-09-17T22:15:53Z", "2026-09-17T22:16:06Z", "9200"),
    (59, "143.88.3.10",  "143.88.4.11",  "2026-09-23T13:25:47Z", "2026-09-23T13:26:03Z", "80"),
    (64, "143.88.6.10",  "143.88.7.11",  "2026-09-23T21:50:00Z", "2026-09-23T21:50:00Z", "80"),
    (65, "143.88.6.10",  "143.88.7.12",  "2026-09-23T22:40:00Z", "2026-09-23T22:40:00Z", "9200"),
]

def epoch(s):
    return datetime.fromisoformat(s.replace("Z", "+00:00")).timestamp()

windows = [("as logged (±2 min)",        0,         120),
           ("+5 h (local entered as UTC)", 5 * 3600, 120),
           ("whole day (±12 h)",          0,         12 * 3600)]

for line, src, dst, start, end, port in checks:
    dst_match = (F.col("dest_ip_zeek").startswith(dst) if dst.endswith(".")
                 else F.col("dest_ip_zeek") == dst)
    pair = df.filter((F.col("src_ip_zeek") == src) & dst_match)
    print(f"\nLine {line}: {src} -> {dst}  logged port {port}  logged {start}")
    for name, shift, pad in windows:
        w = pair.filter(F.col("ts").between(epoch(start) + shift - pad, epoch(end) + shift + pad))
        rows = (w.groupBy("label_technique")
                 .agg(F.count(F.lit(1)).alias("n"),
                      F.collect_set("dest_port_zeek").alias("ports"),
                      F.from_unixtime(F.min("ts").cast("long")).alias("first"))
                 .collect())
        if not rows:
            print(f"  {name:30s} nothing")
        for r in rows:
            ports = ", ".join(sorted(r["ports"], key=lambda p: int(p) if p and p.isdigit() else 0)[:8])
            print(f"  {name:30s} {r['n']:>6} conns  label={r['label_technique']}  ports={ports}  first={r['first']}")